# DEH 30-Day PySpark Challenge
## Day 14 — Semi, Anti & Cross Joins — union & unionByName

**Instructions:**
1. Click `File → Save a copy in Drive` before editing
2. Run each cell in order using `Shift + Enter`
3. Complete the assignment cells at the bottom

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder
    .appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id",       StringType(),  True),
    StructField("customer_id",    StringType(),  True),
    StructField("product_id",     StringType(),  True),
    StructField("order_date",     DateType(),    True),
    StructField("quantity",       IntegerType(), True),
    StructField("unit_price",     DoubleType(),  True),
    StructField("discount_pct",   IntegerType(), True),
    StructField("status",         StringType(),  True),
    StructField("payment_method", StringType(),  True),
    StructField("region",         StringType(),  True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name",  StringType(), True),
    StructField("last_name",   StringType(), True),
    StructField("email",       StringType(), True),
    StructField("city",        StringType(), True),
    StructField("state",       StringType(), True),
    StructField("country",     StringType(), True),
    StructField("signup_date", DateType(),   True),
    StructField("segment",     StringType(), True)
])

orders_df    = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header","true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")
print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()}")

## Step 5 — left semi join

In [ ]:
# Find orders placed by Enterprise customers — order columns only
enterprise_customers = customers_df.filter(F.col("segment") == "Enterprise")

orders_by_enterprise = orders_df.join(
    enterprise_customers,
    on="customer_id",
    how="left_semi"
)

print(f"Enterprise customers : {enterprise_customers.count()}")
print(f"Their orders         : {orders_by_enterprise.count()}")
print(f"Columns in result    : {orders_by_enterprise.columns}")
orders_by_enterprise.show(3)

## Step 6 — left anti join

In [ ]:
# Orders with no matching customer
orders_no_customer = orders_df.join(customers_df, on="customer_id", how="left_anti")
print(f"Orders with no customer match: {orders_no_customer.count()}")

# Customers who never placed an order
customers_no_orders = customers_df.join(orders_df, on="customer_id", how="left_anti")
print(f"Customers with no orders: {customers_no_orders.count()}")
customers_no_orders.select("customer_id", "first_name", "last_name").show()

## Step 7 — cross join

In [ ]:
# All possible region + status combinations
regions  = orders_df.select("region").distinct()
statuses = orders_df.select("status").distinct()

print(f"Regions: {regions.count()} | Statuses: {statuses.count()}")

all_combinations = regions.crossJoin(statuses)
print(f"All combinations: {all_combinations.count()}")
all_combinations.orderBy("region", "status").show()

## Step 8 — union() vs unionByName()

In [ ]:
df1 = spark.createDataFrame([("O0001", "C001", 1299.99)], ["order_id", "customer_id", "unit_price"])
df2 = spark.createDataFrame([("C002", "O0002", 449.99)],  ["customer_id", "order_id", "unit_price"])

print("union() — WRONG (matches by position):")
df1.union(df2).show()

print("unionByName() — CORRECT (matches by name):")
df1.unionByName(df2).show()

In [ ]:
# unionByName with allowMissingColumns
df1 = spark.createDataFrame([("O0001","C001",1299.99,10)], ["order_id","customer_id","unit_price","discount"])
df2 = spark.createDataFrame([("O0002","C002",449.99)],     ["order_id","customer_id","unit_price"])

df1.unionByName(df2, allowMissingColumns=True).show()

---
## Assignment — Day 14

---

### Task 1
Using a **left semi join**, find all orders placed by `SMB` segment customers.  
Only order columns should appear — no customer columns.

In [ ]:
# Task 1 — Write your code here


### Task 2
Using a **left anti join**, find all customers who have never placed an order.  
How many are there in our dataset?

In [ ]:
# Task 2 — Write your code here


### Task 3
Use `crossJoin()` to generate all possible combinations of `region` and `payment_method` from `orders.csv`.  
How many combinations are there?

In [ ]:
# Task 3 — Write your code here


### Task 4
Filter `orders.csv` into two DataFrames — Delivered orders and Cancelled orders.  
Use `unionByName()` to combine them.  
Compare the row counts before and after union.

In [ ]:
# Task 4 — Write your code here


---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*